#### 运行此命令以确保您拥有“智能下载器”（数据集）和 PDF 引擎。


In [21]:
import sys
!{sys.executable} -m pip install datasets pandas fpdf2 tqdm yfinance pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 kB 22.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [datasets]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


#### 环境

In [22]:
import pandas as pd
import os
import ssl
from datasets import load_dataset
from fpdf import FPDF
from tqdm.auto import tqdm

# Fix macOS SSL issues for downloading
ssl._create_default_https_context = ssl._create_unverified_context

print("Environment Ready.")

Environment Ready.


#### 近20年(2004-2024)SPX Index 股票代码清洗 (818 档股票) under ticker.csv


In [23]:
# Load your tickers.csv
df_tickers = pd.read_csv('tickers.csv')

# Clean: Remove #N/A, strip whitespace, keep standard tickers
clean_tickers = df_tickers['tickers'].astype(str).str.strip().str.upper()
clean_tickers = [t for t in clean_tickers if t.isalpha() and 'N/A' not in t]
clean_tickers = sorted(list(set(clean_tickers)))

print(f"Validated {len(clean_tickers)} tickers for the 20-year library.")

Validated 818 tickers for the 20-year library.


In [24]:
print("Pulling the FULL 20-year dataset (all 33,000+ transcripts)...")

# This library handles all 'parts' (shards) automatically
dataset = load_dataset("Bose345/sp500_earnings_transcripts", split="train")

# Convert to a Pandas DataFrame
df_all = dataset.to_pandas()

# Safety check: Handle both 'symbol' or 'ticker' column names
col = 'symbol' if 'symbol' in df_all.columns else 'ticker'

print(f"SUCCESS! You now have {len(df_all)} transcripts in memory.")
print(f"Alphabet Range: {df_all[col].min()} to {df_all[col].max()}")

Pulling the FULL 20-year dataset (all 33,000+ transcripts)...


README.md: 0.00B [00:00, ?B/s]

parquet_files/part-0.parquet:   0%|          | 0.00/1.82G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33362 [00:00<?, ? examples/s]

SUCCESS! You now have 33362 transcripts in memory.
Alphabet Range: A to ZTS


#### 近20年每一季度发布会transcript (32,036pdf 文档) 到 SPX_20yr_PDF_Library_10GB folder

In [25]:
import os
from fpdf import FPDF
from tqdm.auto import tqdm

pdf_output_dir = "SPX_20yr_PDF_Library_10GB"
if not os.path.exists(pdf_output_dir):
    os.makedirs(pdf_output_dir)

# Filter for the tickers in your clean_tickers list
df_final = df_all[df_all['symbol'].isin(clean_tickers)]

print(f"Generating PDFs for {len(df_final)} earnings calls...")

for _, row in tqdm(df_final.iterrows(), total=len(df_final)):
    try:
        date_clean = str(row['date'])[:10]
        filename = f"{row['symbol']}_{date_clean}.pdf"
        filepath = os.path.join(pdf_output_dir, filename)
        
        # Resume Check: Skip if already done
        if os.path.exists(filepath):
            continue
            
        pdf = FPDF()
        pdf.add_page()
        
        # Header
        pdf.set_font("helvetica", "B", 16)
        pdf.cell(0, 10, txt=f"Company: {row['symbol']}", ln=1)
        pdf.set_font("helvetica", "I", 10)
        pdf.cell(0, 8, txt=f"Date: {date_clean}", ln=1)
        pdf.ln(5)
        
        # Body - We use a slightly larger font/spacing to reach 10GB faster
        pdf.set_font("helvetica", size=11)
        content = str(row['content']).encode('latin-1', 'ignore').decode('latin-1')
        # Line height of 7 helps "inflate" the file size for your 10GB goal
        pdf.multi_cell(0, 7, txt=content)
        
        pdf.output(filepath)
    except:
        continue

print(f"Done! Check '{pdf_output_dir}'. You should see the whole alphabet now.")

Generating PDFs for 32038 earnings calls...


  0%|          | 0/32038 [00:00<?, ?it/s]

/var/folders/hj/lv_9mmsd6lgdtkdnv16x25280000gn/T/ipykernel_6745/3885614157.py:29: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(0, 10, txt=f"Company: {row['symbol']}", ln=1)
/var/folders/hj/lv_9mmsd6lgdtkdnv16x25280000gn/T/ipykernel_6745/3885614157.py:29: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 10, txt=f"Company: {row['symbol']}", ln=1)
/var/folders/hj/lv_9mmsd6lgdtkdnv16x25280000gn/T/ipykernel_6745/3885614157.py:31: DeprecationWarning: The parameter "txt" has been renamed to "text" in 2.7.6
  pdf.cell(0, 8, txt=f"Date: {date_clean}", ln=1)
/var/folders/hj/lv_9mmsd6lgdtkdnv16x25280000gn/T/ipykernel_6745/3885614157.py:31: DeprecationWarning: The parameter "ln" is deprecated since v2.5.2. Instead of ln=1 use new_x=XPos.LMARGIN, new_y=YPos.NEXT.
  pdf.cell(0, 8, txt=f"Date: {date_clean}", ln=1)
/var/folders/hj/lv_9mmsd6lgdtkdnv16x25280000gn/T/ipykernel

Done! Check 'SPX_20yr_PDF_Library_10GB'. You should see the whole alphabet now.


#### SPX Index past 20 years tickers OHLC 和交易量批量下载 to csv file (2004-2024)

In [27]:
price_file = "spx_20yr_ohlcv_data.csv"

print(f"Downloading 20 years of OHLCV data for {len(clean_tickers)} stocks...")

# Download in one bulk call to be faster than a loop
# Period 'max' or specific dates ensure we get the full 2004-2024 range
try:
    data = yf.download(clean_tickers, start="2004-01-01", end="2024-12-31", group_by='ticker')
    
    # Restructure from 'Wide' to 'Long' format for better data processing
    df_prices = data.stack(level=0).reset_index()
    df_prices.rename(columns={'level_1': 'Ticker'}, inplace=True)
    
    # Save as a massive CSV
    df_prices.to_csv(price_file, index=False)
    print(f"SUCCESS: Market data saved to {price_file}")
    print(f"Price File Size: {os.path.getsize(price_file) / (1024**2):.2f} MB")
except Exception as e:
    print(f"Error downloading prices: {e}")

$RBK: possibly delisted; no price data found  (1d 2004-01-01 -> 2024-12-31)
[                       1%                       ]  5 of 818 completed$TSS: possibly delisted; no timezone found
[*                      2%                       ]  15 of 818 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
$NBL: possibly delisted; no timezone found
[*                      2%                       ]  16 of 818 completed$TFCFA: possibly delisted; no timezone found
[*                      2%                       ]  17 of 818 completed$ANSS: possibly delisted; no timezone found
[*                      2%                       ]  18 of 818 completed$RTN: possibly delisted; no timezone found
[*                      2%                       ]  19 of 818 completed$MEL: possibly delisted; no price data found  (1d 2004-01-01 -> 2024-12-31)
[*                      3%                       ]  27 of 818 completed$XLNX: 

SUCCESS: Market data saved to spx_20yr_ohlcv_data.csv
Price File Size: 305.50 MB


#### 基本面（i.e. balance sheet, income statement...etc.）by annual quarterly to company_fundamentals folder 

In [29]:
import os
import pandas as pd
import yfinance as yf
from tqdm.auto import tqdm

# Create a folder for the 'Fundamentals Library'
fund_dir = "SPX_Fundamental_History"
if not os.path.exists(fund_dir):
    os.makedirs(fund_dir)

print(f"Extracting all available fundamental history for {len(clean_tickers)} tickers...")

for symbol in tqdm(clean_tickers, desc="Building Fundamental Library"):
    try:
        tk = yf.Ticker(symbol)
        
        # 1. ANNUAL & QUARTERLY STATEMENTS
        # We save everything available to maximize file size/data
        is_df = tk.get_income_stmt(freq='yearly')
        is_q_df = tk.get_income_stmt(freq='quarterly')
        
        bs_df = tk.get_balance_sheet(freq='yearly')
        bs_q_df = tk.get_balance_sheet(freq='quarterly')
        
        cf_df = tk.get_cashflow(freq='yearly')
        cf_q_df = tk.get_cashflow(freq='quarterly')

        # 2. SAVE INDIVIDUAL FILES
        # Each ticker gets its own mini-database in your folder
        is_df.to_csv(f"{fund_dir}/{symbol}_income_annual.csv")
        is_q_df.to_csv(f"{fund_dir}/{symbol}_income_quarterly.csv")
        bs_df.to_csv(f"{fund_dir}/{symbol}_balance_annual.csv")
        bs_q_df.to_csv(f"{fund_dir}/{symbol}_balance_quarterly.csv")
        cf_df.to_csv(f"{fund_dir}/{symbol}_cashflow_annual.csv")
        cf_q_df.to_csv(f"{fund_dir}/{symbol}_cashflow_quarterly.csv")
        
        # 3. COMPANY PROFILE (Metadata)
        # This adds text data like Business Summary, Sector, and Industry
        info = tk.info
        info_df = pd.DataFrame.from_dict(info, orient='index', columns=['Value'])
        info_df.to_csv(f"{fund_dir}/{symbol}_profile_metadata.csv")

    except Exception:
        # If a ticker is delisted or blocked, we just move to the next
        continue

print(f"Fundamental Library Complete. Files stored in: {fund_dir}")

Extracting all available fundamental history for 818 tickers...


Building Fundamental Library:   0%|          | 0/818 [00:00<?, ?it/s]

HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ABKFQ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: ANSS"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: CITGQ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DALRQ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DCNAQ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: DPHIQ"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: FI"}}}
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found fo

Fundamental Library Complete. Files stored in: SPX_Fundamental_History
